# Federated Queries: Neo4j Aircraft Graph + Databricks Sensor Data

Combines graph topology from Neo4j with time-series sensor data from Databricks Delta tables.

**Best-practice pattern:** push graph traversal and graph-side aggregation into `remote_query()`, aggregate Delta tables in Spark, then join the reduced results in one Spark SQL statement. This minimizes rows fetched from Neo4j while keeping the cross-source join visible to Spark.

**Data split:**
- **Neo4j** (graph-native): Aircraft, Systems, Sensors, Flights, Maintenance Events, Delays, all topology and operational data
- **Delta** (tabular): `sensor_readings`, 172,800 hourly readings from 160 sensors over 45 days

**Prerequisites:**
- Run `00-load-graph.ipynb` to load the aircraft graph into Neo4j
- Run `01-neo4j-uc-connection-setup.ipynb` to create the UC JDBC connection

## Graph Schema

Subset of the aircraft graph used by this notebook's federated queries. Every query runs through `remote_query()` (see the next cell). The inner SQL is sent verbatim to the Neo4j JDBC driver, whose SQL-to-Cypher translator rewrites `NATURAL JOIN <RelationshipType>` into Cypher `MATCH (a)-[:REL]->(b)` patterns before execution against Neo4j.

**Nodes**

| Label | Key props used |
|---|---|
| Aircraft | aircraftId, model |
| System | type |
| Sensor | sensorId |
| MaintenanceEvent | aircraftId, severity |
| Flight | operator |
| Delay | cause, minutes |

**Relationships**

| Type | From -> To | Used in |
|---|---|---|
| HAS_SYSTEM | Aircraft -> System | Query 1 |
| HAS_SENSOR | System -> Sensor | Query 1 |

## How federation works in this notebook

Every query in this notebook follows the same shape:

```sql
SELECT ...
FROM remote_query('<uc_connection_name>',
    query => '<SQL sent to Neo4j>'
) AS g
LEFT JOIN <delta_table> AS d ON ...
```

- **`remote_query()`** is a Databricks SQL table-valued function. Databricks sends the inner SQL string verbatim to the foreign source through the UC JDBC connection registered in notebook 01. Databricks itself does not parse or rewrite the inner SQL.
- On the Neo4j side, the **JDBC driver's SQL-to-Cypher translator** parses the SQL string and emits Cypher. `NATURAL JOIN <RelationshipType>` is a connector-specific convention that maps to a Cypher `MATCH (a)-[:REL]->(b)` traversal. Aggregates (`COUNT`, `AVG`, `GROUP BY`) are pushed into Neo4j; only summarized rows cross the wire.
- The Databricks SQL planner then treats the `remote_query()` result as a relation and joins it with Delta tables in a **single statement** — no client-side DataFrame join.

### Why every `GROUP BY` inside `remote_query()` ends with `HAVING COUNT(*) > 0`

You will see `HAVING COUNT(*) > 0` appended to every `GROUP BY` inside `remote_query()` below. Reason:

> Databricks' `remote_query()` has a result-reuse layer that can return an empty result set for pure `GROUP BY` queries on warm clusters (the second and later runs in the same session). The bug is in Databricks' `remote_query()` planning, not in the Neo4j JDBC driver. Adding `HAVING COUNT(*) > 0` is semantically a no-op — every aggregated group has at least one row by definition — but it forces the planner to re-evaluate and bypasses the empty cache.

Documented in `site/modules/ROOT/pages/troubleshooting.adoc`. The inline code comments on each query below refer back to this note rather than repeating the explanation.

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Loaded from Databricks secrets
# =============================================================================

# --- Neo4j Aura ---
SECRET_SCOPE = "neo4j-uc-demos"
NEO4J_URI = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_URI")
NEO4J_USERNAME = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_USERNAME")
NEO4J_PASSWORD = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_PASSWORD")

# --- Databricks Unity Catalog ---
UC_CATALOG = dbutils.secrets.get(scope=SECRET_SCOPE, key="UC_CATALOG")
UC_SCHEMA = dbutils.secrets.get(scope=SECRET_SCOPE, key="UC_SCHEMA")
UC_VOLUME = dbutils.secrets.get(scope=SECRET_SCOPE, key="UC_VOLUME")
JDBC_JAR_PATH = dbutils.secrets.get(scope=SECRET_SCOPE, key="JDBC_JAR_PATH")
UC_CONNECTION_NAME = "sample_neo4j_jdbc_connection"

# =============================================================================
# DERIVED VALUES - no need to edit below this line
# =============================================================================
FQN = f"`{UC_CATALOG}`.`{UC_SCHEMA}`"
VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}"
NEO4J_JDBC_URL_SQL = (
    f"jdbc:{NEO4J_URI}/neo4j"
    "?enableSQLTranslation=true"
    "&timeout=30000"
)
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

print("Configuration:")
print(f"  Neo4j URI:       {NEO4J_URI}")
print(f"  Tables:          {FQN}.*")
print(f"  Volume:          {VOLUME_PATH}")
print(f"  JDBC JAR:        {JDBC_JAR_PATH}")
print(f"  UC Connection:   {UC_CONNECTION_NAME}")

---

## Setup: Load sensor_readings Delta Table

Creates the `sensor_readings` table from the CSV file in the UC Volume.
This table is joined with Neo4j graph data in the federated queries below.

**Schema:** `readingId, sensorId, ts, value`. 172,800 hourly readings from 160 sensors over 45 days.

In [ ]:
print("--- Setup: Load sensor_readings Delta Table ---")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQN}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQN}.`{UC_VOLUME}`")

spark.sql(f"""
    CREATE OR REPLACE TABLE {FQN}.sensor_readings AS
    SELECT
        reading_id AS readingId,
        sensor_id AS sensorId,
        ts,
        value
    FROM read_files('{VOLUME_PATH}/nodes_readings.csv',
        format => 'csv',
        header => true,
        inferColumnTypes => true)
""")

count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {FQN}.sensor_readings").collect()[0]["cnt"]
status = "PASS" if count == 172800 else "FAIL"
print(f"  [{status}] sensor_readings: {count:,} rows")

spark.sql(f"SELECT sensorId, COUNT(*) AS readings FROM {FQN}.sensor_readings GROUP BY sensorId LIMIT 5").show(truncate=False)

---

## Query 1: Sensor Health by Aircraft

A single-statement federated query joining Neo4j graph topology with Delta sensor statistics in pure SQL.

- **`remote_query()` (Neo4j side)**: walks Aircraft → HAS_SYSTEM → System → HAS_SENSOR → Sensor and returns one row per sensor with its owning aircraft and system type. The translator rewrites the `NATURAL JOIN` chain into a Cypher `MATCH` pattern.
- **Delta CTE**: aggregates `sensor_readings` per `sensorId` (count, avg, min, max).
- **Join**: `LEFT JOIN` on `sensorId` inside one `spark.sql()` call. The Databricks planner handles cross-source execution.

**SQL → Cypher example (the `remote_query()` inner SQL):**

| SQL | Cypher |
|-----|--------|
| `SELECT a.aircraftId, a.model, sys.type AS systemType, s.sensorId FROM Aircraft a NATURAL JOIN HAS_SYSTEM r1 NATURAL JOIN System sys NATURAL JOIN HAS_SENSOR r2 NATURAL JOIN Sensor s` | `MATCH (a:Aircraft)-[r1:HAS_SYSTEM]->(sys:System)-[r2:HAS_SENSOR]->(s:Sensor) RETURN a.aircraftId AS aircraftId, a.model AS model, sys.type AS systemType, s.sensorId AS sensorId` |

In [ ]:
print("=" * 60)
print("FEDERATED QUERY 1: Sensor Health by Aircraft")
print("=" * 60)

result = spark.sql(f"""
    WITH neo4j_topology AS (
        SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
            query => 'SELECT a.aircraftId AS aircraftId,
                             a.model AS model,
                             sys.type AS systemType,
                             s.sensorId AS sensorId
                      FROM Aircraft a
                      NATURAL JOIN HAS_SYSTEM r1
                      NATURAL JOIN System sys
                      NATURAL JOIN HAS_SENSOR r2
                      NATURAL JOIN Sensor s'
        )
    ),
    sensor_stats AS (
        SELECT sensorId,
               COUNT(*) AS reading_count,
               ROUND(AVG(value), 2) AS avg_value,
               ROUND(MIN(value), 2) AS min_value,
               ROUND(MAX(value), 2) AS max_value
        FROM {FQN}.sensor_readings
        GROUP BY sensorId
    )
    SELECT t.aircraftId, t.model, t.systemType, t.sensorId,
           s.reading_count, s.avg_value
    FROM neo4j_topology t
    LEFT JOIN sensor_stats s ON s.sensorId = t.sensorId
    ORDER BY t.aircraftId, t.systemType
""")

result.show(10, truncate=False)
row_count = result.count()
status = "PASS" if row_count == 160 else "FAIL"
print(f"  [{status}] {row_count} sensor+aircraft rows")

---

## Query 2: Maintenance Severity and Sensor Health

Two single-statement federated queries that combine Neo4j aggregates with Delta aggregates.

- **`remote_query()` (Neo4j side)**: maintenance event counts grouped by severity, and grouped by aircraft. Aggregation is pushed into Neo4j; only summarized rows cross the wire.
- **Delta CTE**: average sensor value per aircraft, derived from the `AC<n>-` prefix in `sensorId`.
- **Join**: on `aircraftId` inside one `spark.sql()` call.

Each `GROUP BY` inside `remote_query()` ends with `HAVING COUNT(*) > 0` — see the workaround note near the top of the notebook.

**SQL → Cypher examples (the `remote_query()` inner SQL):**

| SQL | Cypher |
|-----|--------|
| `SELECT m.severity AS severity, COUNT(*) AS event_count FROM MaintenanceEvent m GROUP BY m.severity HAVING COUNT(*) > 0` | `MATCH (m:MaintenanceEvent) WITH m.severity AS severity, count(*) AS event_count WHERE event_count > 0 RETURN severity, event_count` |
| `SELECT m.aircraftId AS aircraftId, COUNT(*) AS maint_count FROM MaintenanceEvent m GROUP BY m.aircraftId HAVING COUNT(*) > 0` | `MATCH (m:MaintenanceEvent) WITH m.aircraftId AS aircraftId, count(*) AS maint_count WHERE maint_count > 0 RETURN aircraftId, maint_count` |

In [ ]:
print("=" * 60)
print("FEDERATED QUERY 2: Maintenance Severity + Sensor Health")
print("=" * 60)

# Neo4j-only aggregate via remote_query().
# HAVING COUNT(*) > 0 is the remote_query() GROUP BY workaround (see note at top).
print("\n  Maintenance events by severity:")
spark.sql(f"""
    SELECT severity, event_count
    FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT m.severity AS severity, COUNT(*) AS event_count
                  FROM MaintenanceEvent m
                  GROUP BY m.severity
                  HAVING COUNT(*) > 0'
    )
    ORDER BY event_count DESC
""").show(truncate=False)

# Federated single-statement query: Neo4j aggregate (maintenance per aircraft) joined to
# Delta aggregate (avg sensor reading per aircraft) on aircraftId.
# HAVING COUNT(*) > 0 is the remote_query() GROUP BY workaround (see note at top).
result = spark.sql(f"""
    WITH maint_per_aircraft AS (
        SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
            query => 'SELECT m.aircraftId AS aircraftId, COUNT(*) AS maint_count
                      FROM MaintenanceEvent m
                      GROUP BY m.aircraftId
                      HAVING COUNT(*) > 0'
        )
    ),
    sensor_per_aircraft AS (
        SELECT REGEXP_EXTRACT(sensorId, '^(AC[0-9]+)', 1) AS aircraftId,
               ROUND(AVG(value), 2) AS avg_sensor_reading
        FROM {FQN}.sensor_readings
        GROUP BY REGEXP_EXTRACT(sensorId, '^(AC[0-9]+)', 1)
    )
    SELECT m.aircraftId, m.maint_count, s.avg_sensor_reading
    FROM maint_per_aircraft m
    LEFT JOIN sensor_per_aircraft s ON s.aircraftId = m.aircraftId
    ORDER BY m.maint_count DESC
""")

# Collect once and count locally. A separate result.count() can trigger Spark
# column pruning that wraps the Neo4j aggregate as SELECT "maint_count" FROM (...),
# which the external SQL translator rejects.
rows = result.collect()
print("  Maintenance count + avg sensor reading per aircraft:")
spark.createDataFrame(rows, result.schema).show(10, truncate=False)
count = len(rows)
status = "PASS" if count == 20 else "FAIL"
print(f"  [{status}] {count} aircraft")

---

## Query 3: Flight Delay Analysis by Operator

Pure Neo4j graph analytics via `remote_query()` — no Delta side. Demonstrates aggregate pushdown: the `GROUP BY` runs entirely inside Neo4j and only summarized rows cross the wire.

- **`remote_query()`**: flight counts grouped by operator
- **`remote_query()`**: delay counts and average minutes grouped by cause

Each `GROUP BY` inside `remote_query()` ends with `HAVING COUNT(*) > 0` — see the workaround note near the top of the notebook.

**SQL → Cypher examples (the `remote_query()` inner SQL):**

| SQL | Cypher |
|-----|--------|
| `SELECT f.operator AS operator, COUNT(*) AS flight_count FROM Flight f GROUP BY f.operator HAVING COUNT(*) > 0` | `MATCH (f:Flight) WITH f.operator AS operator, count(*) AS flight_count WHERE flight_count > 0 RETURN operator, flight_count` |
| `SELECT d.cause AS cause, COUNT(*) AS delay_count, AVG(d.minutes) AS avg_minutes FROM Delay d GROUP BY d.cause HAVING COUNT(*) > 0` | `MATCH (d:Delay) WITH d.cause AS cause, count(*) AS delay_count, avg(d.minutes) AS avg_minutes WHERE delay_count > 0 RETURN cause, delay_count, avg_minutes` |

In [ ]:
print("=" * 60)
print("FEDERATED QUERY 3: Flight Delay Analysis by Operator")
print("=" * 60)

# HAVING COUNT(*) > 0 is the remote_query() GROUP BY workaround (see note at top).
print("\n  Flights by operator:")
spark.sql(f"""
    SELECT operator, flight_count
    FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT f.operator AS operator, COUNT(*) AS flight_count
                  FROM Flight f
                  GROUP BY f.operator
                  HAVING COUNT(*) > 0'
    )
    ORDER BY flight_count DESC
""").show(truncate=False)

# HAVING COUNT(*) > 0 is the remote_query() GROUP BY workaround (see note at top).
print("  Delays by cause:")
spark.sql(f"""
    SELECT cause, delay_count, avg_minutes
    FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT d.cause AS cause, COUNT(*) AS delay_count, AVG(d.minutes) AS avg_minutes
                  FROM Delay d
                  GROUP BY d.cause
                  HAVING COUNT(*) > 0'
    )
    ORDER BY delay_count DESC
""").show(truncate=False)

print("Status: PASS. Run 03-materialized-tables.ipynb next")